# **SinBrief Prototype**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers torch scikit-learn networkx
!pip install -q sinling sentencepiece accelerate
!pip install -q gradio reportlab
!pip install -q fpdf2
!pip install -q gradio reportlab pdfplumber
!pip install -q pytesseract pdf2image
!apt-get install -q tesseract-ocr
!apt-get install -q tesseract-ocr-sin

print(" Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.1 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-

In [3]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
from transformers import AutoModel, AutoTokenizer
from sinling import SinhalaTokenizer, POSTagger
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
import warnings
import gradio as gr
import tempfile
import random
from collections import Counter

warnings.filterwarnings('ignore')

print(" Libraries imported")

 Libraries imported


In [4]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/SinBrief"
MODEL_PATH = f"{BASE_PATH}/Model"

# Verify paths
assert os.path.exists(MODEL_PATH), f"Model path not found: {MODEL_PATH}"

print(" Paths configured:")
print(f"   Model: {MODEL_PATH}")

 Paths configured:
   Model: /content/drive/MyDrive/SinBrief/Model


In [5]:
print("Initializing tools...")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Sinhala tools
tokenizer_sin = SinhalaTokenizer()
pos_tagger = POSTagger()

# mBERT tokenizer
tokenizer_mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

print(" Tools initialized")

Initializing tools...
Using device: cuda


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 Tools initialized


In [6]:
# Legal keywords for weighted graphs
LEGAL_KEYWORDS = [
    'අධිකරණය',
    'නීතිය',
    'විනිශ්චය',
    'දඬුවම',
    'චෝදනා',
    'සාක්ෂි',
    'තීන්දුව',
    'අභියාචනය',
    'නඩුව',
    'වරද',
    'දෝෂ',
    'බලය',
    'කොන්දේසි',
    'ප්‍රතිපාදන',
    'ව්‍යවස්ථා',
    'පනත',
    'පැමිණිලිකරු',
    'කොන්ත්‍රාත්තුව',
    'පැමිණිල්ල'
]

print(f" Loaded {len(LEGAL_KEYWORDS)} legal keywords")

 Loaded 19 legal keywords


In [7]:
print("Loading mBERT for sentence embeddings...")

model_mbert = AutoModel.from_pretrained("bert-base-multilingual-cased")
model_mbert.to(device)
model_mbert.eval()

def get_sentence_embedding(sentence):
    """Generate sentence embedding using mBERT"""
    inputs = tokenizer_mbert(sentence, return_tensors="pt",
                             max_length=512, truncation=True,
                             padding=True).to(device)
    with torch.no_grad():
        outputs = model_mbert(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

print(" mBERT loaded for embeddings")

Loading mBERT for sentence embeddings...


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 mBERT loaded for embeddings


In [8]:
def cluster_sentences(sentences, num_clusters=3):
    """
    Cluster sentences by semantic similarity
    """
    if len(sentences) <= num_clusters:
        return [[sent] for sent in sentences]

    # Generate embeddings
    print(f"   Generating embeddings for {len(sentences)} sentences...")
    embeddings = np.array([get_sentence_embedding(sent) for sent in sentences])

    # Cluster
    clustering = AgglomerativeClustering(
        n_clusters=min(num_clusters, len(sentences)),
        metric='cosine',
        linkage='average'
    )
    labels = clustering.fit_predict(embeddings)

    # Group by cluster
    clusters = {}
    for idx, label in enumerate(labels):
        if label not in clusters:
            clusters[label] = []
        clusters[label].append(sentences[idx])

    return list(clusters.values())

print(" Clustering function defined")

 Clustering function defined


In [9]:
KEEP_POS = {'NOUN', 'VERB', 'AUX', 'PROPN', 'ADJ'}

def build_word_graph_weighted(sentences_in_cluster, legal_keywords):
    G = nx.DiGraph()
    G.add_node("START")
    G.add_node("END")

    total_words_added = 0

    for sentence in sentences_in_cluster:
        tokenized = tokenizer_sin.tokenize(sentence)
        if len(tokenized) == 0:
            continue

        # Try POS tagging,
        words_with_pos = []

        try:
            pos_tags = pos_tagger.predict([tokenized])[0]
            words_with_pos = [(w, p) for w, p in zip(tokenized, pos_tags)
                             if p in KEEP_POS and len(w) > 1]
        except:
            pass

        # FALLBACK: If POS tagging failed or gave too few words,
        # use ALL words (filtering only by length and stopwords)
        if len(words_with_pos) < 3:
            # Simple heuristic: keep words longer than 2 characters
            # and prioritize words with legal keywords
            words_with_pos = []
            for w in tokenized:
                if len(w) > 2:  # Skip very short words
                    # Assign dummy POS tag
                    words_with_pos.append((w, 'NOUN'))

        # Limit words per sentence to avoid too dense graphs
        words_with_pos = words_with_pos[:30]

        if len(words_with_pos) == 0:
            continue

        # START → first word
        first_word = words_with_pos[0]
        weight = 2.0 if any(kw in first_word[0] for kw in legal_keywords) else 1.0

        if G.has_edge("START", first_word):
            G["START"][first_word]['weight'] += weight
        else:
            G.add_edge("START", first_word, weight=weight)

        total_words_added += 1

        # Consecutive words
        for i in range(len(words_with_pos)-1):
            current = words_with_pos[i]
            next_word = words_with_pos[i+1]

            # Higher weight if either word is legal keyword
            has_keyword = any(kw in current[0] or kw in next_word[0]
                            for kw in legal_keywords)
            weight = 2.0 if has_keyword else 1.0

            if G.has_edge(current, next_word):
                G[current][next_word]['weight'] += weight
            else:
                G.add_edge(current, next_word, weight=weight)

            total_words_added += 1

        # last → END
        last_word = words_with_pos[-1]
        weight = 2.0 if any(kw in last_word[0] for kw in legal_keywords) else 1.0

        if G.has_edge(last_word, "END"):
            G[last_word]["END"]['weight'] += weight
        else:
            G.add_edge(last_word, "END", weight=weight)

    print(f"       Built graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    return G

print(" FIXED keyword-weighted graph building function defined")

 FIXED keyword-weighted graph building function defined


In [10]:
def generate_fused_sentences(G, num_paths=5, max_length=25, min_length=8):
    if G.number_of_nodes() <= 2:
        return []

    candidates = []
    seen_word_sets = set()  # Avoid duplicate word combinations

    # Method 1: Weighted random walks (favor high-weight edges)
    for attempt in range(num_paths * 5):  # More attempts for better candidates
        try:
            path = ["START"]
            current = "START"
            words = []

            # Random walk with length constraints
            for step in range(max_length + 5):  # Allow some overshoot
                neighbors = list(G.successors(current))

                # Remove END from neighbors unless we have enough words
                if len(words) < min_length and "END" in neighbors:
                    neighbors.remove("END")

                if len(neighbors) == 0:
                    break

                # If we have enough words and END is available
                if len(words) >= min_length and "END" in neighbors:
                    if random.random() < 0.3:  # 30% chance to end
                        break

                # Weighted selection (prefer higher weights)
                weights = []
                valid_neighbors = []
                for n in neighbors:
                    if n != "END":
                        weight = G[current][n].get('weight', 1.0)
                        weights.append(weight)
                        valid_neighbors.append(n)

                if len(valid_neighbors) == 0:
                    break

                # Pick next node with weighted probability
                next_node = random.choices(valid_neighbors, weights=weights)[0]

                # Extract word
                if isinstance(next_node, tuple):
                    word = next_node[0]
                    if len(word) > 1:  # Skip single chars
                        words.append(word)

                current = next_node

                # Stop if we reach max length
                if len(words) >= max_length:
                    break

            # Check for have valid length
            if len(words) < min_length or len(words) > max_length:
                continue

            # Check for duplicates (same set of words)
            word_set = frozenset(words)
            if word_set in seen_word_sets:
                continue

            seen_word_sets.add(word_set)

            # Create sentence
            fused = ' '.join(words)

            # Quality checks
            if len(fused) < 20:  # Too short
                continue

            # Check for excessive repetition
            unique_words = len(set(words))
            if unique_words < len(words) * 0.6:  # More than 40% repetition
                continue

            candidates.append(fused)

            if len(candidates) >= num_paths:
                break

        except Exception as e:
            continue

    # Method 2: If not enough, use weighted word extraction
    if len(candidates) < num_paths:
        # Get words sorted by importance (edge weight)
        word_weights = {}

        for u, v, data in G.edges(data=True):
            for node in [u, v]:
                if isinstance(node, tuple) and node not in ["START", "END"]:
                    word = node[0]
                    if len(word) > 1:
                        weight = data.get('weight', 1.0)
                        word_weights[word] = word_weights.get(word, 0) + weight

        if len(word_weights) >= min_length:
            # Sort by weight
            sorted_words = sorted(word_weights.items(), key=lambda x: x[1], reverse=True)

            # Create different combinations
            attempts = 0
            while len(candidates) < num_paths and attempts < 10:
                attempts += 1

                # Take top weighted words with some randomness
                n_words = random.randint(min_length, min(max_length, len(sorted_words)))

                if attempts == 1:
                    # First attempt: pure top words
                    selected = [w for w, _ in sorted_words[:n_words]]
                else:
                    # Later attempts: mix high and medium weight words
                    top_k = len(sorted_words) // 2
                    top_words = [w for w, _ in sorted_words[:top_k]]
                    selected = random.sample(top_words, min(n_words, len(top_words)))

                # Check uniqueness
                word_set = frozenset(selected)
                if word_set in seen_word_sets:
                    continue

                seen_word_sets.add(word_set)

                fused = ' '.join(selected)

                # Quality check
                if len(fused) >= 20:
                    candidates.append(fused)

    return candidates[:num_paths]  # Return at most num_paths

print(" Path generation function defined")

 Path generation function defined


In [11]:
def filter_low_quality_candidates(candidates):
    """
    Filter out obviously bad candidates
    """
    filtered = []

    for candidate in candidates:
        # Skip if too short
        if len(candidate) < 20:
            continue

        # Skip if too many repeated words
        words = candidate.split()
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.6:  # More than 40% repetition
            continue

        # Skip if same word appears 3+ times
        word_counts = Counter(words)
        max_count = max(word_counts.values())
        if max_count >= 3:
            continue

        # All checks passed
        filtered.append(candidate)

    return filtered if len(filtered) > 0 else candidates  # Return original if all filtered

print(" Quality filtering function defined")

 Quality filtering function defined


In [12]:
print("Loading trained sentence scorer...")

class LegalSentenceScorer(nn.Module):
    def __init__(self, model_name="bert-base-multilingual-cased", dropout=0.3):
        super(LegalSentenceScorer, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        score = self.classifier(cls_embedding)
        return score.squeeze()

# Initialize and load trained model
scorer_model = LegalSentenceScorer()
checkpoint = torch.load(f"{MODEL_PATH}/legal_sentence_scorer.pt", map_location=device)
scorer_model.load_state_dict(checkpoint['model_state_dict'])
scorer_model.to(device)
scorer_model.eval()

print(f" Scorer model loaded (Val Accuracy: {checkpoint['best_val_accuracy']:.4f})")

Loading trained sentence scorer...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Scorer model loaded (Val Accuracy: 0.8363)


In [13]:
def generate_summary_with_scoring(document, num_clusters=3):
    """
    Generate abstractive summary with improved quality
    IMPROVEMENTS: Longer output, multiple sentences per cluster, complete endings
    """
    # Split into sentences
    import re
    split_pattern = re.compile(r'[.!?;]+')
    sentences = [s.strip() for s in split_pattern.split(document)
                if len(s.strip()) > 20]

    if len(sentences) < 3:
        return '. '.join(sentences) + '.'

    # Step 1: Cluster sentences
    print(f"   Clustering {len(sentences)} sentences...")
    clusters = cluster_sentences(sentences, num_clusters=min(3, max(2, len(sentences)//5)))
    print(f"   Created {len(clusters)} clusters")

    summary_sentences = []

    # Step 2: Process each cluster - SELECT TOP 2 CANDIDATES (not just 1!)
    for cluster_idx, cluster in enumerate(clusters, 1):
        if len(cluster) == 0:
            continue

        print(f"   Processing cluster {cluster_idx}/{len(clusters)} ({len(cluster)} sentences)...")

        # Build keyword-weighted graph
        G = build_word_graph_weighted(cluster, LEGAL_KEYWORDS)

        if G.number_of_nodes() <= 10:
            print(f"       Graph has only {G.number_of_nodes()} nodes, skipping")
            continue

        # Generate MORE candidates with LONGER length
        candidates = generate_fused_sentences(G, num_paths=8, max_length=25, min_length=8)

        if len(candidates) == 0:
            print(f"      No candidates generated, skipping cluster")
            continue

        # Filter low quality
        candidates = filter_low_quality_candidates(candidates)

        if len(candidates) == 0:
            print(f"       All candidates filtered out, skipping cluster")
            continue

        print(f"     Generated {len(candidates)} abstractive candidates, scoring...")

        # Score ALL candidates
        candidate_scores = []

        for cand_idx, candidate in enumerate(candidates):
            encoding = tokenizer_mbert(
                candidate,
                add_special_tokens=True,
                max_length=128,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

            with torch.no_grad():
                score = scorer_model(
                    encoding['input_ids'].to(device),
                    encoding['attention_mask'].to(device)
                ).item()

            candidate_scores.append((candidate, score))
            print(f"       Candidate {cand_idx+1}: {score:.4f} | {candidate[:60]}...")

        # Sort by score
        candidate_scores.sort(key=lambda x: x[1], reverse=True)

        # Take TOP 2 candidates per cluster (not just 1!)
        top_k = min(2, len(candidate_scores))

        for i in range(top_k):
            candidate, score = candidate_scores[i]
            print(f"     Selected #{i+1} (score: {score:.4f})")
            summary_sentences.append(candidate)

    # Step 3: Combine with smart formatting
    if len(summary_sentences) == 0:
        return '. '.join(sentences[:3]) + '.'

    # Join all selected sentences
    summary = '. '.join(summary_sentences)

    # Step 4: Ensure proper sentence ending
    target_min = 300
    target_max = 500

    if len(summary) < target_min:
        # Too short - add period
        if not summary.endswith('.'):
            summary += '.'

    elif len(summary) <= target_max:
        # Perfect length - just ensure period
        if not summary.endswith('.'):
            summary += '.'

    else:
        # Too long - truncate at sentence boundary

        # Try to find last complete sentence before target_max
        sentences_in_summary = [s.strip() for s in summary.split('.') if s.strip()]

        # Build summary by adding sentences until we exceed target
        final_summary = ""
        for sent in sentences_in_summary:
            test_summary = final_summary + sent + '. '

            if len(test_summary) > target_max:
                # This sentence would make it too long
                if len(final_summary) > target_min:
                    # We already have enough content, stop here
                    break
                else:
                    # We need this sentence even if it's a bit long
                    final_summary = test_summary
                    break
            else:
                final_summary = test_summary

        summary = final_summary.strip()

        # Ensure ends with period
        if not summary.endswith('.'):
            # Find last complete word
            last_space = summary.rfind(' ')
            if last_space > target_min:
                summary = summary[:last_space] + '.'
            else:
                summary += '.'

    # Final safety: always end with period
    if not summary.endswith('.'):
        summary += '.'

    return summary

print(" Summary generation with longer output defined")

 Summary generation with longer output defined


In [14]:
def extract_text_from_pdf(file_path):
    """
    Extract text from PDF.
    First tries pdfplumber (fast, works for digital PDFs).
    If result is garbled or empty, falls back to OCR using tesseract.
    """
    import pdfplumber

    # --- Attempt 1: pdfplumber (digital PDF) ---
    text_parts = []
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text_parts.append(page_text)
    except Exception as e:
        print(f"   pdfplumber failed: {e}")

    full_text = "\n".join(text_parts).strip()

    # Check if extracted text looks valid (Sinhala unicode range: \u0D80-\u0DFF)
    sinhala_chars = sum(1 for c in full_text if '\u0D80' <= c <= '\u0DFF')
    total_chars = len(full_text)

    print(f"   pdfplumber extracted {total_chars} chars, {sinhala_chars} Sinhala chars")

    if total_chars > 100 and sinhala_chars > 10:
        print("   Using pdfplumber result")
        return full_text

    # --- Attempt 2: OCR with tesseract ---
    print("   Text looks garbled or empty, trying OCR...")
    try:
        import pytesseract
        from pdf2image import convert_from_path

        pages = convert_from_path(file_path, dpi=300)
        ocr_parts = []

        for i, page_img in enumerate(pages):
            print(f"   OCR processing page {i+1}/{len(pages)}...")
            # Use Sinhala + English language pack
            ocr_text = pytesseract.image_to_string(
                page_img,
                lang='sin+eng',
                config='--psm 6'
            )
            if ocr_text.strip():
                ocr_parts.append(ocr_text.strip())

        ocr_full = "\n".join(ocr_parts).strip()
        print(f"   OCR extracted {len(ocr_full)} characters")

        if len(ocr_full) > 100:
            return ocr_full
        else:
            return full_text  # return whatever we had even if poor

    except Exception as e:
        print(f"   OCR failed: {e}")
        return full_text  # return whatever pdfplumber got

print(" Smart PDF extraction with OCR fallback defined")

 Smart PDF extraction with OCR fallback defined


In [15]:
def create_pdf(summary_text):
    """Create a PDF file containing the summary"""
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont
        from reportlab.lib.units import inch

        font_path = "/content/NotoSansSinhala.ttf"
        if not os.path.exists(font_path):
            print("   Downloading Sinhala font...")
            import urllib.request
            urllib.request.urlretrieve(
                "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansSinhala/NotoSansSinhala-Regular.ttf",
                font_path
            )
            print("   Font downloaded.")

        pdfmetrics.registerFont(TTFont('NotoSansSinhala', font_path))

        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix='.pdf')

        doc = SimpleDocTemplate(temp_pdf.name, pagesize=letter)
        story = []

        styles = getSampleStyleSheet()

        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=16,
            textColor='blue',
            spaceAfter=30,
            alignment=1
        )

        body_style = ParagraphStyle(
            'SinhalaBody',
            parent=styles['BodyText'],
            fontName='NotoSansSinhala',
            fontSize=12,
            leading=20,
        )

        story.append(Paragraph("SinBrief - Document Summary", title_style))
        story.append(Spacer(1, 0.2*inch))
        story.append(Paragraph(summary_text, body_style))

        doc.build(story)

        return temp_pdf.name

    except Exception as e:
        print(f"PDF generation error: {e}")
        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w', encoding='utf-8')
        temp_pdf.write(summary_text)
        temp_pdf.close()
        return temp_pdf.name

print(" PDF generation function defined")

 PDF generation function defined


In [16]:
import csv
import datetime

# Create feedback directory
FEEDBACK_PATH = "/content/drive/MyDrive/SinBrief/Prototype/Feedback"
FEEDBACK_FILE = f"{FEEDBACK_PATH}/user_feedback.csv"

os.makedirs(FEEDBACK_PATH, exist_ok=True)

# Initialize CSV if it doesn't exist
if not os.path.exists(FEEDBACK_FILE):
    with open(FEEDBACK_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Timestamp', 'User_Name', 'Document_Filename', 'Rating', 'Feedback_Text'])

def save_feedback(user_name, document_name, rating, feedback_text):
    """Save user feedback to CSV file"""
    try:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # Handle empty values
        user_name = user_name if user_name else "Anonymous"
        rating = rating if rating else "No rating"
        feedback_text = feedback_text if feedback_text else "No feedback"
        document_name = document_name if document_name else "Unknown document"

        # Append to CSV
        with open(FEEDBACK_FILE, 'a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([timestamp, user_name, document_name, rating, feedback_text])

        return " Thank you! Your feedback has been saved."
    except Exception as e:
        return f" Error saving feedback: {str(e)}"

print(" Feedback system initialized")

 Feedback system initialized


In [17]:
def process_document(file):
    """
    Processes an uploaded .txt or .pdf document and returns a summary.
    Returns a clear error message for any unsupported file format.
    """
    if file is None:
        return "Please upload a document file.", None, None, None, ""

    file_path = file if isinstance(file, str) else file.name
    document_filename = os.path.basename(file_path)

    try:
        with open(file_path, 'rb') as f:
            header = f.read(5)
        if header.startswith(b'%PDF-'):
            ext = '.pdf'
        else:
            ext = '.txt'
    except Exception as e:
        return f" Could not read file: {str(e)}", None, None, None, ""

    try:
        if ext == '.pdf':
            print(f"   Extracting text from PDF: {document_filename}")
            document_text = extract_text_from_pdf(file_path)
        else:
            with open(file_path, 'r', encoding='utf-8') as f:
                document_text = f.read().strip()

        print(f"   Document length: {len(document_text)} characters")

        if not document_text or len(document_text) < 50:
            return (
                " Could not extract enough text from the document. "
                "If this is a scanned PDF, OCR may not have recognised the Sinhala text.",
                document_text or "", None, None, document_filename
            )

        print("\n" + "="*60)
        print("GENERATING SUMMARY")
        print("="*60 + "\n")

        summary = generate_summary_with_scoring(document_text, num_clusters=3)

        txt_file = tempfile.NamedTemporaryFile(delete=False, suffix='.txt',
                                               mode='w', encoding='utf-8')
        txt_file.write(summary)
        txt_file.close()

        pdf_file = create_pdf(summary)

        print("\n" + "="*60)
        print("SUMMARY GENERATION COMPLETE")
        print(f"Summary length: {len(summary)} characters")
        print("="*60)

        return summary, document_text, txt_file.name, pdf_file, document_filename

    except Exception as e:
        error_msg = f" Error processing document: {str(e)}"
        print(error_msg)
        import traceback; traceback.print_exc()
        return error_msg, None, None, None, ""

print(" process_document defined")

 process_document defined


In [18]:
theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="green",
)

with gr.Blocks(theme=theme, title="SinBrief - Legal Document Summarizer") as demo:

    gr.Markdown(
        """
        # SinBrief
        ### Abstractive Summarization for Sinhala Legal Documents

        Upload a Sinhala legal document (**.txt** or **.pdf**) to generate an abstractive summary.
        """
    )

    document_filename_state = gr.State("")

    with gr.Row():
        with gr.Column(scale=1):
            file_input = gr.File(
                label="Upload Document (.txt or .pdf)",
                file_types=[".txt", ".pdf"],
                type="filepath"
            )

            generate_btn = gr.Button("Generate Summary", variant="primary", size="lg")

            document_preview = gr.Textbox(
                label="Uploaded Document Preview",
                lines=10,
                placeholder="Your document will appear here...",
                interactive=False
            )

        with gr.Column(scale=1):
            summary_output = gr.Textbox(
                label="Generated Summary",
                lines=15,
                placeholder="Summary will appear here...",
                interactive=False
            )

            with gr.Row():
                txt_download = gr.DownloadButton(label="⬇ Download as TXT", visible=False)
                pdf_download = gr.DownloadButton(label="⬇ Download as PDF", visible=False)

    gr.Markdown("---")

    gr.Markdown(
        """
        ## Feedback
        Help us improve! Please share your thoughts on the summary quality.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            user_name_input = gr.Textbox(
                label="Your Name (Optional)",
                placeholder="Enter your name...",
                lines=1
            )
            rating_input = gr.Slider(
                minimum=1,
                maximum=5,
                step=1,
                label="Rating (1-5 stars)",
                value=3
            )

        with gr.Column(scale=1):
            feedback_input = gr.Textbox(
                label="Feedback (Optional)",
                placeholder="Share your thoughts on the summary quality...",
                lines=4
            )

    submit_feedback_btn = gr.Button("Submit Feedback", variant="secondary")
    feedback_status = gr.Textbox(label="Status", interactive=False, visible=True)

    def process_and_update(file):
        summary, preview, txt_path, pdf_path, filename = process_document(file)
        txt_visible = txt_path is not None
        pdf_visible = pdf_path is not None
        return (
            summary,
            preview,
            gr.DownloadButton(value=txt_path, visible=txt_visible),
            gr.DownloadButton(value=pdf_path, visible=pdf_visible),
            filename
        )

    generate_btn.click(
        fn=process_and_update,
        inputs=[file_input],
        outputs=[summary_output, document_preview, txt_download, pdf_download, document_filename_state]
    )

    submit_feedback_btn.click(
        fn=save_feedback,
        inputs=[user_name_input, document_filename_state, rating_input, feedback_input],
        outputs=[feedback_status]
    )

print("\n" + "="*60)
print("LAUNCHING SINBRIEF INTERFACE")
print("="*60 + "\n")

demo.launch(share=True, debug=True)


LAUNCHING SINBRIEF INTERFACE

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://73b60bdcd2ce6e5e50.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


   Document length: 10704 characters

GENERATING SUMMARY

   Clustering 65 sentences...
   Generating embeddings for 65 sentences...
   Created 3 clusters
   Processing cluster 1/3 (54 sentences)...
       Built graph: 496 nodes, 874 edges
     Generated 8 abstractive candidates, scoring...
       Candidate 1: 0.2818 | මෙහි මින් මතු මුදල යනුවෙන් සඳුන්වනු ලබන තුරු බලාත්මක නොවන්නේ...
       Candidate 2: 0.3381 | අරමුදලේ යම් ආගමික සංස්කෘතික අරමුදල් පනත හින්දු සංස්කෘතික අරම...
       Candidate 3: 0.2656 | රීති වලට යටත්ව හින්දු ආගම සංස්කෘතිය වැඩි දියුණු කිරීම...
       Candidate 4: 0.3791 | අරමුදලේ ප්‍රධාන පරිපාලන නිලධරයා විය යුතු අතර දෙදෙනා පැමිණ සි...
       Candidate 5: 0.3125 | පනත යටතේ රැස්කරන ලද්දා මුදල් විෂයය භාර අමාත්‍යවරයාගේ එකඟත්වය...
       Candidate 6: 0.3791 | සාමාජිකයින් අතුරෙන් කෙනෙකු මණ්ඩලයේ රැස්වීමක් සඳහා අවශ්‍ය යයි...
       Candidate 7: 0.3791 | ආදායම් වියදම් ගිණුමක් පළමු සඳහන් ගෙවීම් කරනු ලැබිය යුතු අතර ...
       Candidate 8: 0.3734 | පනත යටතේ රැස්කරන ලද්දා මුදල් වෙනත් 